# Setup môi trường
1. Truy cập [Google AI Studio](https://aistudio.google.com/apikey) và chọn `Create API Key`
2. Tạo file `.env` và lưu API key dưới dạng `GOOGLE_API_KEY="YOUR_API_KEY"`
3. Sử dụng thư viện `python-dotenv` để quản lý API Key

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin\AppData\Local\Temp\ipykernel_13276\86987914.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=google_api_key)

# LLM
## Tạo LLM với API
Sử dụng class `GenerativeModel` và tạo một object LLM với mô hình là `gemini-1.5-flash`

In [6]:
MODEL_VERSION = "gemini-2.5-flash"
model = genai.GenerativeModel(MODEL_VERSION)

## Tương tác với LLM
Thử tương tác với mô hình bằng phương thức `generate_content` của đối tượng `model`

In [7]:
prompt = "What is LLM?"
response = model.generate_content(prompt)
response

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "**LLM stands for Large Language Model.**\n\nIt's a type of artificial intelligence (AI) program designed to understand, generate, and process human language. LLMs are trained on massive datasets of text and code, allowing them to learn patterns, grammar, facts, and even some reasoning capabilities.\n\nLet's break down what each part of the name means:\n\n1.  **Large:**\n    *   **Data:** These models are trained on truly enormous amounts of text data \u2013 billions or even trillions of words gathered from the internet (web pages, books, articles, forums), code repositories, and more.\n    *   **Parameters:** They have billions, sometimes hundreds of billions, of parameters (the weights and biases in their neural network) which are adjusted during training. 

Kết quả sẽ trả về một đối tượng có cấu trúc, và ta quan sát được câu trả lời của mô hình nằm ở phần `result` -> `candidates` -> `content` -> `parts` -> `text`.

Để truy cập nhanh câu trả lời, sử dụng trực tiếp thuộc tính `text` của đối tượng `response`.

In [8]:
response.text

'**LLM stands for Large Language Model.**\n\nIt\'s a type of artificial intelligence (AI) program designed to understand, generate, and process human language. LLMs are trained on massive datasets of text and code, allowing them to learn patterns, grammar, facts, and even some reasoning capabilities.\n\nLet\'s break down what each part of the name means:\n\n1.  **Large:**\n    *   **Data:** These models are trained on truly enormous amounts of text data – billions or even trillions of words gathered from the internet (web pages, books, articles, forums), code repositories, and more.\n    *   **Parameters:** They have billions, sometimes hundreds of billions, of parameters (the weights and biases in their neural network) which are adjusted during training. This vast number of parameters allows them to capture extremely complex relationships and nuances in language.\n\n2.  **Language:**\n    *   Their primary domain is **natural language** – the way humans communicate through speaking an

## Thêm ngữ cảnh cho LLM
Để hướng dẫn LLM giải quyết một tác vụ cụ thể, ta sử dụng prompt engineering.

Bạn là chủ một nhà hàng. Hãy viết hướng dẫn phù hợp để LLM của bạn có thể:
1. quảng cáo về nhà hàng
2. giới thiệu menu cho khách hàng

Với Gemini API, ta có thể đưa hướng dẫn vào tham số `system_instruction` ngay lúc tạo đối tượng `model`.

In [9]:
model = genai.GenerativeModel(MODEL_VERSION,
                              system_instruction="""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: phở, gỏi cuốn, cơm tấm, bún bò.
                              Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng để biết thêm thông tin.
                              """)

Thử lại với prompt đến từ khách hàng.

In [11]:
prompt = "Menu"
response = model.generate_content(prompt)
response.text

'Chào mừng quý khách đến với Viet Cuisine!\n\nMenu của nhà hàng chúng tôi gồm có các món ăn đặc sắc sau:\n*   **Phở**\n*   **Gỏi cuốn**\n*   **Cơm tấm**\n*   **Bún bò**\n\nChúc quý khách có một bữa ăn ngon miệng!'

## Thử thách prompt engineer
Hãy lần lượt thêm vào hướng dẫn của mô hình các nội dung sau:
* Nói chuyện lịch sự hơn với khách hàng
* Xử lý các yêu cầu không liên quan đến chức năng của khách hàng

Theo dõi cách mô hình thay đổi câu trả lời khi đã chỉnh sửa hướng dẫn.

In [15]:
model = genai.GenerativeModel(MODEL_VERSION,
                              system_instruction="""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: phở, gỏi cuốn, cơm tấm, bún bò.
                              Ngoài hai chức năng trên, bạn không hỗ trợ chức năng nào khác. Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp.'
                              Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khách hàng, vì khách hàng là thượng đế.
                              """)

In [16]:
prompt = "Menu"
response = model.generate_content(prompt)
response.text

'Chào bạn! Rất vui được hỗ trợ bạn. Menu của nhà hàng Viet Cuisine của chúng tôi gồm có các món ăn đặc sắc sau:\n*   **Phở**\n*   **Gỏi cuốn**\n*   **Cơm tấm**\n*   **Bún bò**\n\nBạn có muốn tìm hiểu thêm về món nào không ạ?'

In [17]:
prompt = "Tôi muốn hủy bàn."
response = model.generate_content(prompt)
response.text

'Chào bạn, tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp nhé. Cảm ơn bạn!'

## Kết nối file dữ liệu với LLM

Đọc file dữ liệu từ `menu.csv` vào DataFrame`menu_df`

In [18]:
menu_df = pd.read_csv('menu.csv', index_col=[0])
menu_df

,name,description,ingredients,notes
0,Gỏi Cuốn,Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá...,"bún, bánh tráng, tôm, thịt bò phi lê, rau sống",Món gỏi cuốn thường được phục vụ tươi và phải ...
1,Phở Việt Nam,Nổi tiếng với hương vị đậm đà và hương thơm củ...,"bún phở, thịt bò, thịt gà, hành tây, hành phi,...",Thịt bò có thể chọn giữa tái và chín.
2,Cơm Tấm,Cơm tấm là một món ăn đường phố phổ biến trong...,"gạo tấm, thịt heo, trứng, chả, dưa leo, nước m...",Cơm tấm thường được ăn vào bữa trưa hoặc bữa t...
3,Bún Bò,Bún bò là một món ăn đặc trưng của ẩm thực miề...,"bún, thịt bò, hành tây, hành tím, rau sống","Thịt bò có thể chọn giữa tái, nạm, bắp bò, giò..."
3,Khoai Tây Chiên,Khoai tây chiên là một món ăn phổ biến và được...,"khoai tây, dầu, muối",NaN


Cập nhật hướng dẫn với cột `name` trong `menu_df` và thử lại với prompt mới.

In [19]:
model = genai.GenerativeModel(MODEL_VERSION,
                              system_instruction=f"""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: {','.join(menu_df["name"].to_list())}.
                              3. Lịch mở cửa của nhà hàng: Từ T2 -> T6: từ 9:30 sáng đến 9:30 tối, T7 -> CN: 7:00 sáng đến 10:30 tối.
                              Ngoài hai chức năng trên, bạn không hỗ trợ chức năng nào khác. Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp.'
                              Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khách hàng, vì khách hàng là thượng đế.
                              """)

In [20]:
from IPython.display import Markdown

prompt = "Liệt kê các món ăn trong menu"
answer = model.generate_content(prompt)
Markdown(answer.text)

Chào bạn, rất vui được hỗ trợ! Dưới đây là các món ăn đặc sắc trong menu của nhà hàng Viet Cuisine của chúng tôi:

*   Gỏi Cuốn
*   Phở Việt Nam
*   Cơm Tấm
*   Bún Bò
*   Khoai Tây Chiên

Chúc bạn có một bữa ăn ngon miệng tại Viet Cuisine nhé!

In [21]:
prompt = "Giới thiệu các món ăn trong menu"
answer = model.generate_content(prompt)
Markdown(answer.text)

Chào bạn! Rất vui được giới thiệu các món ăn đặc sắc trong menu của nhà hàng Viet Cuisine ạ.

Chúng tôi có:
*   **Gỏi Cuốn:** Món khai vị tươi ngon, thanh mát.
*   **Phở Việt Nam:** Món phở truyền thống chuẩn vị Việt.
*   **Cơm Tấm:** Đặc sản cơm tấm Sài Gòn thơm ngon, đậm đà.
*   **Bún Bò:** Món bún bò cay nồng, hấp dẫn.
*   **Khoai Tây Chiên:** Món ăn vặt quen thuộc, giòn tan.

Bạn muốn tìm hiểu thêm về món nào không ạ?